In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.models import stacking_ensemble
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [ ]:
raw_path = os.path.dirname(os.path.abspath('.'))

save_probas = "..\probas"

df_FE_train_path = os.path.join(raw_path, "df_FE_train.csv")
df_FE_test_path = os.path.join(raw_path, "df_FE_test.csv")
df_FE = pd.read_csv(df_FE_train_path)
df_FE_test = pd.read_csv(df_FE_test_path)

X = df_FE.drop(['Stress_Level'], axis=1)
y = df_FE['Stress_Level']

y_test = df_FE_test['Stress_Level']

In [ ]:
from imblearn.over_sampling import SMOTE
target_names = ['0', '1']
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X,y)

In [ ]:
oof_rmlp_probas = np.load(os.path.join(save_probas, 'oof_RMLP_y_probas.npy'))
oof_tm_probas = np.load(os.path.join(save_probas, 'oof_TM_y_probas.npy'))
oof_etc_probas = np.load(os.path.join(save_probas, 'oof_ETC_y_probas.npy'))
oof_xgb_probas = np.load(os.path.join(save_probas, 'oof_XGB_y_probas.npy'))
oof_lgbm_probas = np.load(os.path.join(save_probas, 'oof_LGBM_y_probas.npy'))
oof_nn_probas = np.load(os.path.join(save_probas, 'oof_nn_probas.npy'))

rmlp_test_probas = np.load(os.path.join(save_probas, 'RMLP_test_probas.npy'))
tm_test_probas = np.load(os.path.join(save_probas, 'TM_test_probas.npy'))
etc_test_probas = np.load(os.path.join(save_probas, 'ETC_test_probas.npy'))
xgb_test_probas = np.load(os.path.join(save_probas, 'XGB_test_probas.npy'))
lgbm_test_probas = np.load(os.path.join(save_probas, 'LGBM_test_probas.npy'))
nn_test_probas = np.load(os.path.join(save_probas, 'nn_test_probas.npy'))

In [ ]:
oof_probas_list = [oof_rmlp_probas, oof_tm_probas, oof_etc_probas, oof_xgb_probas, oof_lgbm_probas, oof_nn_probas]
test_probas_list = [rmlp_test_probas, tm_test_probas, etc_test_probas, xgb_test_probas, lgbm_test_probas, nn_test_probas]

for list in oof_probas_list:
    print(list.shape)

for list in test_probas_list:
    print(list.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(
    max_iter = 3000,
    C=0.01,
    random_state=42,
)

In [ ]:
preds, probas, model = stacking_ensemble(oof_probas_list=oof_probas_list,
                                         y=y_resampled,
                                         test_probas_list=test_probas_list,
                                         meta_model=meta_model)



submission = pd.DataFrame({
    'class': preds,
})

submission.to_csv("..\preds\stacking_submission.csv", index=False)
print('Done')

In [ ]:
from src.utils import show_confusion_matrix
from sklearn.metrics import classification_report

for probas in test_probas_list:
    past_preds = probas.argmax(axis=1)

    show_confusion_matrix(y_test, past_preds, target_names)
    print(classification_report(y_test, past_preds))
    print(f"{'='*60}")


print(f"Final evaluation")
show_confusion_matrix(y_test, preds, target_names)
print(classification_report(y_test, preds))